# CodeTune v2 — DPO Training (Self-Contained)

**Environment**: Google Colab A100 80GB  
**Goal**: DPO on top of sft_C. Fully self-contained — no external file uploads needed.  
**DPO pairs**: Generated on Colab with 4-bit sft_C + MiniMax API scoring.

| Cell | Task | Est. Time |
|------|------|-----------|
| 1–3 | Setup & login | 5 min |
| 4 | Download sft_C + prompts from HF Hub | 10 min |
| 5 | Generate DPO pairs (200 prompts × 2 temps) | ~25 min |
| 6 | Stats | instant |
| 7 | Free GPU memory | instant |
| 8 | DPO training (beta=0.1) | ~35 min |
| 9 | Sanity check on HE/54 | 2 min |
| 10 | Upload to HF Hub | 5 min |
| 11 | (Optional) Beta ablation | — |

In [ ]:
# Cell 1 — GPU check
import subprocess, torch
r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                   capture_output=True, text=True)
print("GPU:", r.stdout.strip())
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB, BF16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Cell 2 — Install (unsloth MUST be first)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes -q
!pip install datasets huggingface-hub openai tenacity -q

# Import unsloth first to suppress "should be imported before transformers" warning
import unsloth  # noqa: F401

# Patch trl availability flags so optional packages (mergekit, weave, llm_blender)
# don't cause ImportError. These flags are module-level constants set at import time
# via importlib — patching sys.modules does NOT work, must patch the flags directly.
import trl.import_utils as _trl_iu
for _flag in ["_mergekit_available", "_weave_available", "_llm_blender_available"]:
    if hasattr(_trl_iu, _flag):
        setattr(_trl_iu, _flag, False)

print("Done.")

In [ ]:
# Cell 3 — Credentials & Drive mount
from huggingface_hub import login
from google.colab import drive
import os

HF_TOKEN         = "YOUR_HF_TOKEN"
HF_USERNAME      = "Michlitt"
MINIMAX_API_KEY  = "YOUR_MINIMAX_API_KEY"
MINIMAX_BASE_URL = "https://api.minimax.io/v1"
MINIMAX_MODEL    = "MiniMax-M2.5"
WANDB_API_KEY    = "YOUR_WANDB_API_KEY"   # ← 可选

login(token=HF_TOKEN)
print("Logged in as", HF_USERNAME)

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    REPORT_TO = "wandb"
    print("W&B logged in")
else:
    os.environ["WANDB_DISABLED"] = "true"
    REPORT_TO = "none"
    print("W&B disabled")

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/codetune"
print(f"Drive mounted. DRIVE_ROOT={DRIVE_ROOT}")

In [ ]:
# Cell 4 — Download sft_C adapter + SFT train prompts from HF Hub
from huggingface_hub import snapshot_download, hf_hub_download
from pathlib import Path
import json

# sft_C LoRA adapter
Path("sft_C").mkdir(exist_ok=True)
snapshot_download(
    repo_id=f"{HF_USERNAME}/codetune-v2-sft-C",
    local_dir="sft_C",
    repo_type="model",
)
print("sft_C downloaded")

# SFT train prompts — used as DPO candidate generation source
Path("data/processed").mkdir(parents=True, exist_ok=True)
loc = hf_hub_download(
    repo_id=f"{HF_USERNAME}/codetune-v2-sft",
    filename="sft_train.jsonl",
    repo_type="dataset",
    local_dir="data/processed",
)

# Robust JSONL reading: some lines may contain unescaped newlines
all_samples, bad = [], 0
for l in Path(loc).read_text(encoding="utf-8").splitlines():
    if not l.strip():
        continue
    try:
        all_samples.append(json.loads(l))
    except json.JSONDecodeError:
        bad += 1

if bad:
    print(f"WARNING: skipped {bad} malformed lines (unescaped newlines in data)")
print(f"Prompts loaded: {len(all_samples):,}")

In [ ]:
# Cell 5 — Load DPO pairs from Drive
# v2 pairs generated by 07_dpo_v2_data_generation.ipynb (450 train + 50 val)
# Falls back to v1 exec pairs if v2 not found.
import json, shutil
from pathlib import Path

def _try_load(drive_dir, train_name, val_name, label):
    t = Path(DRIVE_ROOT) / drive_dir / train_name
    v = Path(DRIVE_ROOT) / drive_dir / val_name
    if t.exists() and v.exists():
        return t, v, label
    return None

source = (
    _try_load("dpo_v2_pairs", "dpo_pairs_v2.jsonl",   "dpo_pairs_v2_val.jsonl",   "v2") or
    _try_load("dpo_exec_pairs", "dpo_pairs_exec.jsonl", "dpo_pairs_exec_val.jsonl", "v1 (fallback)")
)
if source is None:
    raise FileNotFoundError(
        "No DPO pairs found in Drive. Run 07_dpo_v2_data_generation.ipynb first."
    )

drive_train, drive_val, pairs_label = source
Path("data/processed").mkdir(parents=True, exist_ok=True)
local_train = Path("data/processed/dpo_pairs_active.jsonl")
local_val   = Path("data/processed/dpo_pairs_active_val.jsonl")
shutil.copy(drive_train, local_train)
shutil.copy(drive_val,   local_val)

train_n = sum(1 for l in drive_train.read_text(encoding="utf-8").splitlines() if l.strip())
val_n   = sum(1 for l in drive_val.read_text(encoding="utf-8").splitlines() if l.strip())
print(f"Using DPO pairs [{pairs_label}]: train={train_n}  val={val_n}")
print(f"Source: {drive_train}")

In [ ]:
# Cell 6 — Pair stats
import json, textwrap
from pathlib import Path
from collections import Counter

train = [json.loads(l) for l in Path("data/processed/dpo_pairs_active.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
val   = [json.loads(l) for l in Path("data/processed/dpo_pairs_active_val.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
src   = Counter(p.get("source","?") for p in train + val)

print(f"Train: {len(train)}  Val: {len(val)}")
print("By source:", dict(src))
if train:
    p = train[0]
    print(f"\n─── Sample pair (source={p['source']}) ───")
    print("Prompt  :", textwrap.shorten(p["prompt"], 100))
    print("Chosen  :", textwrap.shorten(p["chosen"], 100))
    print("Rejected:", textwrap.shorten(p["rejected"], 100))

In [ ]:
# Cell 7 — Free GPU memory before DPO training
# (Only needed if Cell 5 triggered generation; safe to run regardless)
import gc, torch

for _var in ["sft_model", "sft_processor", "sft_tok", "mm_client"]:
    try:
        del globals()[_var]
    except KeyError:
        pass

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory reserved: {torch.cuda.memory_reserved(0)/1e9:.1f} GB")

In [ ]:
# Cell 8 — DPO Training (dpo_v2)
# ~60 min on A100 80GB (450 pairs × 3 epochs)
import json, sys, torch
from pathlib import Path

# ── Patch trl BEFORE importing dpo_trainer ───────────────────────────────────
import trl.import_utils as _trl_iu
_trl_iu._mergekit_available = False
_trl_iu._weave_available = False

from trl.trainer.dpo_trainer import DPOTrainer
from trl.trainer.dpo_config import DPOConfig
from datasets import Dataset
from transformers import AutoTokenizer
from transformers.models.auto.modeling_auto import MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES
from unsloth import FastLanguageModel

# ── Config ────────────────────────────────────────────────────────────────────
EXP_ID       = "dpo_v2"
BETA         = 0.1
LR           = 5e-5
EPOCHS       = 3
MAX_LEN      = 1536
MAX_PROMPT   = 512
BATCH        = 2
GRAD_ACCUM   = 8
WARMUP_STEPS = 30   # 450対 × 3 epochs → ~84 steps, warmup ~35%
OUT_DIR      = f"results/dpo_checkpoints/{EXP_ID}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────────
def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

train_raw = load_jsonl("data/processed/dpo_pairs_active.jsonl")
val_raw   = load_jsonl("data/processed/dpo_pairs_active_val.jsonl")
steps_per_epoch = len(train_raw) // (BATCH * GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS
print(f"Train: {len(train_raw)} pairs  Val: {len(val_raw)} pairs")
print(f"Steps per epoch: {steps_per_epoch}  Total steps: {total_steps}")

# ── Load sft_C (bf16) ─────────────────────────────────────────────────────────
print("\nLoading sft_C (bf16) ...")

if "qwen3_5" not in MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES:
    MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES["qwen3_5"] = "Qwen3_5ForConditionalGeneration"

model, _ = FastLanguageModel.from_pretrained(
    model_name="sft_C", max_seq_length=MAX_LEN,
    load_in_4bit=False, dtype=torch.bfloat16,
)
FastLanguageModel.for_training(model)

tokenizer = AutoTokenizer.from_pretrained("sft_C")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.__class__ = type(
    "PatchedTokenizer",
    (type(tokenizer),),
    {"tokenizer": property(lambda self: self)},
)

n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n/1e6:.1f}M")

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

# ── Pop qwen3_5 AFTER loading, BEFORE DPOTrainer ─────────────────────────────
MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES.pop("qwen3_5", None)

# ── Datasets ──────────────────────────────────────────────────────────────────
_TARGETED_PREFIX = "Complete the following Python function:\n\n"

_COLON_RE = __import__("re").compile(r"^( *):[ ]?", __import__("re").MULTILINE)

def _clean_body(t): return _COLON_RE.sub(r"", t)

def fmt(row):
    chosen   = _clean_body(row["chosen"])
    rejected = _clean_body(row["rejected"])
    if row.get("source", "").startswith("targeted"):
        raw_prompt = row["prompt"].replace(_TARGETED_PREFIX, "", 1)
        if chosen.startswith(raw_prompt):
            chosen = chosen[len(raw_prompt):]
        if rejected.startswith(raw_prompt):
            rejected = rejected[len(raw_prompt):]
    return {"prompt": row["prompt"], "chosen": chosen, "rejected": rejected}

train_ds = Dataset.from_list(train_raw).map(fmt)
val_ds   = Dataset.from_list(val_raw).map(fmt)

# ── Train ─────────────────────────────────────────────────────────────────────
print(f"\nStarting DPO (exp={EXP_ID}, beta={BETA}, epochs={EPOCHS}, steps≈{total_steps}) ...")
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=DPOConfig(
        output_dir=OUT_DIR,
        beta=BETA,
        loss_type="sigmoid",
        max_prompt_length=MAX_PROMPT,
        max_length=MAX_LEN,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=WARMUP_STEPS,
        bf16=True, tf32=True,
        optim="adamw_8bit",
        seed=42,
        logging_steps=1,
        eval_steps=10,
        save_steps=20,
        save_total_limit=2,
        report_to=REPORT_TO,
        run_name=EXP_ID,
    ),
)
trainer.train()

# ── Save ──────────────────────────────────────────────────────────────────────
final_dir = f"{OUT_DIR}/final"
Path(final_dir).mkdir(exist_ok=True)
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"\nSaved to {final_dir}")

In [ ]:
# Cell 9 — Sanity check: HE/54 (same_chars) — persistent sft_C failure
import gc, torch
from transformers.models.auto.modeling_auto import MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES

for _v in ["model", "tokenizer"]:
    try:
        del globals()[_v]
    except KeyError:
        pass
gc.collect()
torch.cuda.empty_cache()

from unsloth import FastLanguageModel

# Restore qwen3_5 entry if Cell 8 already popped it
if "qwen3_5" not in MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES:
    MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES["qwen3_5"] = "Qwen3_5ForConditionalGeneration"

m, processor = FastLanguageModel.from_pretrained(
    f"results/dpo_checkpoints/{EXP_ID}/final",
    max_seq_length=512, load_in_4bit=True,
)
FastLanguageModel.for_inference(m)

MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES.pop("qwen3_5", None)

tok = processor.tokenizer if hasattr(processor, "tokenizer") else processor

PROBE = """def same_chars(s0: str, s1: str):
    \"\"\"Check if two words have the same characters.
    >>> same_chars('eabcdzzzz', 'dddzzzzzzzddeddabc')
    True
    >>> same_chars('abcd', 'dddddddabc')
    True
    >>> same_chars('eabcd', 'dddddddabc')
    False
    \"\"\"
"""

text = tok.apply_chat_template(
    [{"role": "system", "content": "You are an expert Python programmer."},
     {"role": "user",   "content": f"Complete the following Python function:\n\n{PROBE}"}],
    tokenize=False, add_generation_prompt=True,
)
input_ids = tok(text=text, return_tensors="pt", truncation=True, max_length=512)["input_ids"].to(m.device)
with torch.no_grad():
    out = m.generate(input_ids, max_new_tokens=128, do_sample=False,
                     pad_token_id=tok.eos_token_id)
result = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
print("HE/54 completion:")
print(result)
print()
if "set(s0)" in result and "set(s1)" in result:
    print("PASS ✓  —  uses set() correctly")
elif "sorted" in result:
    print("FAIL ✗  —  still uses sorted() (DPO did not fix this case)")
else:
    print("UNKNOWN — check completion manually")

In [ ]:
# Cell 10 — Upload DPO checkpoint to HF Hub
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

DPO_REPO = f"{HF_USERNAME}/codetune-v2-dpo"
try:
    api.create_repo(repo_id=DPO_REPO, repo_type="model", private=True)
    print("Created:", DPO_REPO)
except Exception as e:
    print("Repo already exists:", e)

api.upload_folder(
    folder_path=f"results/dpo_checkpoints/{EXP_ID}/final",
    repo_id=DPO_REPO,
    repo_type="model",
)
print(f"\nUploaded DPO model → {DPO_REPO}")

## Cell 11 — Optional: Beta Ablation

Run this only if Cell 8 results are unsatisfactory.

| Beta | When to use |
|------|-------------|
| `0.05` | `rewards/chosen` is good but HumanEval barely improved — go more aggressive |
| `0.1` | Default starting point |
| `0.3` | `rewards/chosen` declined during training — add stronger KL penalty |

To rerun: change `BETA` and `EXP_ID` in Cell 8 (e.g. `EXP_ID = "dpo_v2_beta03"`, `BETA = 0.3`),  
then re-run Cell 8 → 9 → 10.